# 00. Quickstart

CPU quickstart: graph, solve, independent cut check, Tools.

In [1]:
import json, tempfile
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from QA_max_cut_lib import WeightedGraph, QAOACPU, ExactCPU, tools
ROOT=Path.cwd()
if not (ROOT/'configs').exists(): ROOT=ROOT.parent
graph=WeightedGraph([(10,30,1.5),(30,80,-0.5),(10,80,2.0)], nodes=[10,30,80,101])
reference=ExactCPU(); ref_solution=reference.solve(graph, report=0); print('reference',ref_solution,reference.cut_value)

reference {10: 1, 30: -1, 80: -1, 101: 1} 3.5


## QAOA configuration and first solve

In [2]:
config={'seed':42,'parameters':{'layers':2},'objective':{'type':'mean_cut'},
 'estimation':{'mode':'exact'},'optimizer':{'parameters':{'max_steps':3}},
 'readout':{'mode':'most_probable'},'limits':{'shots_total':0},
 'collect':{'optimizer_history':True,'probabilities':True}}
solver=QAOACPU(config); solution=solver.solve(graph, report=0, logs=0)
print('solution',solution); print('cut',solver.cut_value,'energy',solver.solution_energy)
assert solver.cut_value==graph.cut_value(solution)
assert {int(k): int(v) for k,v in solver.result['solution'].items()}==solution

solution {10: 1, 30: 1, 80: 1, 101: 1}
cut 0.0 energy -0.0


## Tools: graph and optimization trace

In [3]:
fig=tools.plot_graph_solution(solver); plt.close(fig)
fig=tools.plot_optimization(solver); plt.close(fig)
print('probability entries',len(solver.probabilities))

probability entries 16


## Second solve on a changed graph

In [4]:
changed=graph.copy(); changed.set_weight(10,30,0.25)
new_solution=solver.solve(changed, report=0, logs=0)
print('new cut',solver.cut_value); print('previous result unchanged',solver.result['graph']['fingerprint']!=graph.snapshot()['fingerprint'])

new cut -0.25
previous result unchanged True


## Save and no-overwrite

In [5]:
from QA_max_cut_lib.core.saving import save_result
out=Path(tempfile.mkdtemp(prefix='qa_quickstart_'))
record=solver.last_run
path=save_result(record, {'path':str(out),'filename':'quickstart','atts':['solution'],'save_original_data':[]})
print('saved',path)
try:
 save_result(record, {'path':str(out),'filename':'quickstart','atts':['solution'],'save_original_data':[]})
except Exception as exc:
 print('no-overwrite:',type(exc).__name__)

saved SavedFiles(primary=WindowsPath('C:/Users/Alexey/AppData/Local/Temp/qa_quickstart_a4fknf85/quickstart.json'), files=(WindowsPath('C:/Users/Alexey/AppData/Local/Temp/qa_quickstart_a4fknf85/quickstart.json'),))
no-overwrite: FileExistsError


## GPU

GPU cells are intentionally not run in this CPU quickstart. See `docs/GPU.md`; G8 remains a release gate.